In [ ]:
%pip install -q boto3 sagemaker pandas numpy

# Week 3 Tuesday -- AWS Bedrock Foundations and Governance

On Wednesday you built attention and transformer architectures from scratch. You know what BERT, GPT, and T5 are -- encoder-only, decoder-only, and encoder-decoder transformers trained on billions of tokens. Today you use those same architectures, scaled to production, through **AWS Bedrock**: managed API access to foundation models with no infrastructure to manage.

By the end of this notebook you will have:

1. **Configured** boto3 clients for Bedrock and explored available foundation models.
2. **Invoked** a foundation model for text generation using the Messages API.
3. **Generated fraud investigation summaries** from raw transaction data for FraudShield.
4. **Compared** two models on the same task (quality, latency, cost).
5. **Implemented** streaming responses for interactive applications.
6. **Created and tested** a guardrail with content filtering and PII redaction.

| Block | Content |
|-------|---------|
| Stage 1 | Bedrock Foundations: Console Overview, Boto3 Setup, Foundation Models, Model Invocation |
| Stage 2 | Building with Bedrock: FraudShield Summaries, Model Comparison, Streaming |
| Stage 3 | Governance and Guardrails: IAM, Content Filtering, PII Redaction, Cost, Monitoring |

**Pairing callout:**

- **Wednesday (W2):** You built attention and transformers from scratch in NumPy. You used Hugging Face to run BERT, GPT-2, and DistilBART. Bedrock provides managed access to production-scale versions of these same transformer architectures -- Claude (decoder-only, like GPT), Titan (various architectures), Llama (decoder-only), Mistral (decoder-only). The difference is scale: billions of parameters, trained on trillions of tokens, hosted by AWS.

## Setup: AWS Connection and Bedrock Clients

Run the cells below to establish your AWS connection and create the two Bedrock clients.

Bedrock uses **two separate clients**:

| Client | Service Name | Purpose |
|--------|-------------|--------|
| `bedrock` | `bedrock` | Management: list models, create guardrails, manage access |
| `bedrock-runtime` | `bedrock-runtime` | Inference: invoke models, stream responses |

This is the same management/runtime split as SageMaker (`sagemaker` vs `sagemaker-runtime`).

In [ ]:
import boto3
import json
import time
from datetime import datetime

region = boto3.session.Session().region_name or "us-east-1"

bedrock = boto3.client("bedrock", region_name=region)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=region)

sts = boto3.client("sts")
identity = sts.get_caller_identity()

TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")

print("AWS connection verified")
print(f"  Account:  {identity['Account']}")
print(f"  Region:   {region}")
print(f"  Bedrock client:         ready")
print(f"  Bedrock Runtime client:  ready")
print(f"  Session timestamp:       {TIMESTAMP}")

---
# STAGE 1 -- Bedrock Foundations

**Connecting to Wednesday:** On Wednesday you built transformers from scratch and used Hugging Face to run pre-trained models locally. Today you access production-scale transformers through a managed API. No model weights to download, no GPU instances to provision, no containers to configure.

**What you will learn:**
- What Bedrock is and what it manages for you
- How to list and filter available foundation models
- How to apply model selection criteria
- How to invoke a foundation model using the Messages API

## What is AWS Bedrock

Bedrock is a fully managed service that provides **API access to foundation models** from multiple providers. You do not train these models. You do not host them. You do not manage GPU instances. You call an API, send a prompt, and get a response.

| Aspect | Description |
|--------|------------|
| **What it is** | Managed API access to foundation models (LLMs, embedding models, image generators) |
| **What you do NOT manage** | Infrastructure, model weights, scaling, GPU instances, containers |
| **What you DO control** | Which model to invoke, prompt design, guardrails, IAM policies, cost limits |
| **Pricing** | Pay per input/output token (on-demand) or reserved capacity (provisioned throughput) |
| **Providers** | Anthropic (Claude), Amazon (Titan), Meta (Llama), Mistral, Cohere, AI21, Stability AI |

**How it connects to what you already know:**

| Concept | SageMaker (Weeks 1-2) | Bedrock (Today) |
|---------|----------------------|----------------|
| Model source | You train it | Pre-trained by the provider |
| Infrastructure | You provision instances | Fully managed |
| Invocation | `sagemaker-runtime.invoke_endpoint()` | `bedrock-runtime.invoke_model()` |
| Customization | Full control (algorithm, hyperparameters) | Prompt engineering, optional fine-tuning |
| Billing | Per instance-hour | Per token |

> **Discussion:** When would you still train your own model on SageMaker instead of using Bedrock? (Domain-specific tasks where prompting is insufficient, structured prediction like XGBoost fraud scoring, cost efficiency at very high volume, or strict latency requirements.)

## Console Overview (Conceptual)

The Bedrock console provides a graphical interface for managing model access, testing prompts, and configuring governance. We work through the API today, but you should understand the console layout.

| Console Section | Purpose |
|----------------|--------|
| **Model access** | Request and manage access to foundation models from each provider |
| **Playgrounds** | Interactive testing -- send prompts to models and see responses in real time |
| **Guardrails** | Create and manage content filtering and PII redaction policies |
| **Model catalog** | Browse all available models with descriptions, pricing, capabilities |
| **Custom models** | Fine-tune or continue pre-training foundation models on your own data |
| **Provisioned throughput** | Reserve dedicated model capacity for consistent performance |

**Important:** Before you can invoke a model through the API, you must **request access** in the console under Model Access. Some models approve instantly; others require review. If you get `AccessDeniedException` later in this notebook, check model access first.

## Create Bedrock and Bedrock Runtime Clients

We already created the clients in the setup cell. This section explains the two-client pattern.

- **`bedrock`** (management client): Use for listing models, creating guardrails, managing provisioned throughput. Think of it as the SageMaker `create_*` and `describe_*` equivalent.
- **`bedrock-runtime`** (inference client): Use for invoking models and streaming responses. Think of it as the `sagemaker-runtime.invoke_endpoint()` equivalent.

> **What is happening in the next cell:** We verify the clients work by calling `list_foundation_models()` on the management client. This returns all models available in your account and region.

In [ ]:
response = bedrock.list_foundation_models()
models = response["modelSummaries"]
print(f"Total foundation models available: {len(models)}")
print(f"\nFirst 3 model IDs:")
for m in models[:3]:
    print(f"  {m['modelId']}")

## List Available Foundation Models

Bedrock provides access to models from multiple providers. The `list_foundation_models()` API returns metadata about each model: name, provider, modalities, and the model ID you use for invocation.

> **What is happening in the next cell:** We list all available foundation models and display them as a formatted table showing the model name, provider, input/output modalities, and model ID. The model ID is the identifier you pass to `invoke_model()`.

In [ ]:
import pandas as pd

model_data = []
for m in models:
    model_data.append({
        "Model Name": m.get("modelName", "N/A"),
        "Provider": m.get("providerName", "N/A"),
        "Input Modalities": ", ".join(m.get("inputModalities", [])),
        "Output Modalities": ", ".join(m.get("outputModalities", [])),
        "Model ID": m.get("modelId", "N/A"),
    })

models_df = pd.DataFrame(model_data)

print(f"Available Foundation Models ({len(models_df)} total)")
print("=" * 100)

text_models = models_df[
    models_df["Input Modalities"].str.contains("TEXT", na=False)
    & models_df["Output Modalities"].str.contains("TEXT", na=False)
].head(15)

print(f"\nText-to-Text models (first 15):")
print(text_models.to_string(index=False))

providers = models_df["Provider"].value_counts()
print(f"\nModels by provider:")
for provider, count in providers.items():
    print(f"  {provider}: {count}")

## Model Selection Criteria

With dozens of models available, choosing the right one requires evaluating four dimensions:

| Criterion | What to Evaluate | FraudShield Example |
|-----------|-----------------|---------------------|
| **Latency** | Time to first token, total generation time | Real-time fraud alerts need low latency -- choose Haiku or Mistral |
| **Cost** | Price per 1K input/output tokens | High-volume batch summaries -- choose Titan or Haiku for cost efficiency |
| **Capability** | Reasoning depth, instruction following, output quality | Complex investigation summaries -- choose Sonnet or Opus |
| **Context Window** | Maximum input + output tokens | Long transaction histories -- models with 100K+ context |

**Selection heuristic:**

```
Need the highest quality reasoning?
  YES --> Claude Sonnet or Opus
  NO  --> Is latency critical (< 1 second)?
            YES --> Claude Haiku or Mistral
            NO  --> Is cost the primary concern?
                      YES --> Titan Text or Haiku
                      NO  --> Claude Sonnet (balanced)
```

**The rule of thumb:** Start with the smallest/cheapest model that might work. Only upgrade if quality is insufficient. Do not default to the most expensive model.

> **Discussion:** FraudShield wants to generate one-paragraph investigation summaries from transaction data. The summaries are reviewed by human analysts, not shown to customers. Which model would you start with, and why?

## Invoke a Foundation Model -- Text Generation

This is the core operation: sending a prompt to a foundation model and getting a response. We use the **Messages API** format, which is the standard for Claude models on Bedrock.

**Request components:**

| Parameter | Purpose | Typical Value |
|-----------|---------|---------------|
| `modelId` | Which foundation model to invoke | `anthropic.claude-3-haiku-20240307-v1:0` |
| `messages` | Conversation (role + content pairs) | `[{"role": "user", "content": "..."}]` |
| `max_tokens` | Maximum tokens to generate | 256-4096 depending on task |
| `temperature` | Randomness (0 = deterministic, 1 = creative) | 0.1-0.3 for factual tasks |
| `anthropic_version` | API version for Anthropic models | `bedrock-2023-05-31` |

> **What is happening in the next cell:** We invoke Claude 3 Haiku with a simple prompt about fraud detection. We then parse the response to extract the generated text and token usage. Token counts directly determine cost.

In [ ]:
MODEL_ID = "anthropic.claude-3-haiku-20240307-v1:0"

request_body = json.dumps({
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 256,
    "temperature": 0.2,
    "messages": [
        {
            "role": "user",
            "content": "Explain in 3 sentences what transaction fraud detection involves and why machine learning is useful for it."
        }
    ]
})

start_time = time.time()
response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    contentType="application/json",
    accept="application/json",
    body=request_body,
)
latency = time.time() - start_time

response_body = json.loads(response["body"].read())

generated_text = response_body["content"][0]["text"]
input_tokens = response_body["usage"]["input_tokens"]
output_tokens = response_body["usage"]["output_tokens"]
stop_reason = response_body["stop_reason"]

print(f"Model: {MODEL_ID}")
print(f"Latency: {latency:.2f}s")
print(f"Stop reason: {stop_reason}")
print(f"Tokens -- input: {input_tokens}, output: {output_tokens}")
print(f"\nGenerated text:")
print(f"  {generated_text}")

## Request/Response Format

The Claude Messages API uses a conversation format:

**Request:**
```json
{
  "anthropic_version": "bedrock-2023-05-31",
  "max_tokens": 512,
  "temperature": 0.2,
  "messages": [
    {"role": "user", "content": "Your prompt here"}
  ]
}
```

**Response:**
```json
{
  "content": [{"type": "text", "text": "Model response here"}],
  "usage": {"input_tokens": 42, "output_tokens": 128},
  "stop_reason": "end_turn"
}
```

**Stop reasons:**

| `stop_reason` | Meaning | Action |
|---------------|---------|--------|
| `end_turn` | Model finished naturally | Output is complete |
| `max_tokens` | Hit the `max_tokens` limit | Output was truncated -- increase the limit |
| `stop_sequence` | Hit a custom stop sequence | Output stopped at your specified boundary |

Each model provider has a slightly different request schema. Amazon Titan uses `inputText` instead of `messages`. Mistral uses a different format. The Bedrock documentation specifies the schema for each model.

---
# STAGE 2 -- Building with Bedrock

**Connecting to the FraudShield scenario:** Over two weeks you built a fraud detection pipeline: synthetic data, feature engineering, model training, deployment, monitoring. Now we add a new capability: **generating human-readable investigation summaries** from raw transaction data. This is the kind of task foundation models excel at -- converting structured data into natural language narratives.

**What you will learn:**
- How to design structured prompts for data-to-text tasks
- How to compare models on quality, latency, and cost
- When to use Bedrock vs SageMaker for LLM workloads
- How streaming responses work and when to use them

## FraudShield Use Case: Generate Investigation Summaries

FraudShield's fraud analysts currently write investigation summaries manually. Each summary requires reviewing raw transaction data, identifying suspicious patterns, and writing a narrative that explains why the transaction was flagged. This takes **15-20 minutes per case**.

The pipeline we build today:

```
Raw Transaction Data (amount, hour, merchant, risk score, flag reason)
    |
    v
Structured Prompt (instructions + data + output format)
    |
    v
Foundation Model (Bedrock API)
    |
    v
Draft Investigation Summary (human analyst reviews and approves)
```

The model does **not** make the final fraud decision. It generates a draft that a human analyst reviews. This is the "human in the loop" pattern -- the model accelerates the analyst's work without replacing their judgment.

> **Discussion:** What could go wrong if we used the model's summary directly without human review? (Hallucinated details, misinterpreted risk scores, missed context, legal liability.)

## Structured Prompt for Fraud Investigation Summaries

Prompt quality determines output quality. We construct a structured prompt with four components:

| Component | Purpose | Example |
|-----------|---------|--------|
| **System context** | Tell the model its role | "You are a fraud investigation analyst..." |
| **Data section** | Provide raw transaction data | Amount, hour, merchant, risk score |
| **Instructions** | Specify what output you want | "Generate a summary with: risk assessment, key indicators, recommended action" |
| **Format constraints** | Control output structure | "Use bullet points. Keep to 3-4 sentences per section." |

> **What is happening in the next cell:** We define a transaction record, construct a structured prompt, and invoke Claude to generate an investigation summary. Pay attention to how specific the instructions are -- specificity in the prompt produces specificity in the output.

In [ ]:
transaction = {
    "transaction_id": "TXN-2024-0847291",
    "amount": 2847.50,
    "hour": 3,
    "merchant_name": "Overseas Electronics Direct",
    "merchant_category": "Electronics",
    "merchant_risk_score": 0.91,
    "distance_from_home_km": 4200,
    "transaction_count_24h": 12,
    "is_international": True,
    "card_present": False,
    "model_fraud_probability": 0.94,
    "flag_reason": "High amount + international + high merchant risk + unusual hour",
}

prompt = f"""You are a fraud investigation analyst at FraudShield Risk Analytics. Your job is to review flagged transactions and generate concise investigation summaries for human analysts.

Below is a flagged transaction. Generate an investigation summary with the following sections:

1. **Risk Assessment**: One sentence stating the overall risk level (Low / Medium / High / Critical) and why.
2. **Key Indicators**: Bullet points listing each suspicious factor from the data.
3. **Contextual Analysis**: 2-3 sentences explaining what pattern this transaction fits (e.g., card-not-present fraud, account takeover, friendly fraud).
4. **Recommended Action**: One sentence stating what the analyst should do next.

Transaction Data:
- Transaction ID: {transaction['transaction_id']}
- Amount: ${transaction['amount']:,.2f}
- Time: {transaction['hour']}:00 (local time)
- Merchant: {transaction['merchant_name']} ({transaction['merchant_category']})
- Merchant Risk Score: {transaction['merchant_risk_score']}
- Distance from Home: {transaction['distance_from_home_km']:,} km
- Transactions in Last 24h: {transaction['transaction_count_24h']}
- International: {transaction['is_international']}
- Card Present: {transaction['card_present']}
- Model Fraud Probability: {transaction['model_fraud_probability']}
- Flag Reason: {transaction['flag_reason']}

Generate the investigation summary now."""

request_body = json.dumps({
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 1024,
    "temperature": 0.2,
    "messages": [
        {"role": "user", "content": prompt}
    ]
})

start_time = time.time()
response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    contentType="application/json",
    accept="application/json",
    body=request_body,
)
latency = time.time() - start_time

response_body = json.loads(response["body"].read())
summary = response_body["content"][0]["text"]
usage = response_body["usage"]

print(f"Model: {MODEL_ID}")
print(f"Latency: {latency:.2f}s")
print(f"Tokens -- input: {usage['input_tokens']}, output: {usage['output_tokens']}")
print(f"\n{'=' * 70}")
print("GENERATED INVESTIGATION SUMMARY")
print(f"{'=' * 70}")
print(summary)

## Compare Two Models on the Same Task

One of Bedrock's advantages is **model choice**. You can test multiple models on the same task and compare quality, speed, and cost. We run the same fraud summary prompt through two different models.

> **What is happening in the next cell:** We invoke a second model (Amazon Titan Text Express) with the same transaction data and prompt structure, then compare the outputs side by side. Note that Titan uses a different request format than Claude.

In [ ]:
TITAN_MODEL_ID = "amazon.titan-text-express-v1"

titan_request_body = json.dumps({
    "inputText": prompt,
    "textGenerationConfig": {
        "maxTokenCount": 1024,
        "temperature": 0.2,
        "topP": 0.9,
    }
})

start_time = time.time()
try:
    titan_response = bedrock_runtime.invoke_model(
        modelId=TITAN_MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=titan_request_body,
    )
    titan_latency = time.time() - start_time
    titan_response_body = json.loads(titan_response["body"].read())
    titan_summary = titan_response_body["results"][0]["outputText"]
    titan_token_count = titan_response_body["results"][0].get("tokenCount", "N/A")

    print(f"{'=' * 70}")
    print(f"MODEL COMPARISON: Same prompt, different models")
    print(f"{'=' * 70}")
    print(f"\n--- Claude 3 Haiku ---")
    print(f"Latency: {latency:.2f}s")
    print(f"Tokens: input={usage['input_tokens']}, output={usage['output_tokens']}")
    print(f"Summary (first 300 chars):")
    print(f"  {summary[:300]}...")
    print(f"\n--- Amazon Titan Text Express ---")
    print(f"Latency: {titan_latency:.2f}s")
    print(f"Output tokens: {titan_token_count}")
    print(f"Summary (first 300 chars):")
    print(f"  {titan_summary[:300]}...")
    print(f"\n--- Comparison Notes ---")
    print(f"Compare: output quality, detail level, structure, actionability.")
    print(f"In production, run this comparison on 50-100 samples with domain expert review.")

except Exception as e:
    print(f"Titan invocation failed: {e}")
    print(f"This may indicate Titan model access has not been enabled.")
    print(f"To proceed, enable Titan Text Express in the Bedrock console under Model Access.")
    print(f"\nClaude summary is still available from the previous cell.")

## Managed vs Self-hosted: Bedrock vs SageMaker for LLMs

You have now used both SageMaker endpoints (Weeks 1-2) and Bedrock (today). When should you use each?

| Dimension | Bedrock (Managed) | SageMaker (Self-hosted) |
|-----------|-------------------|------------------------|
| **Infrastructure** | None -- fully managed | You provision and manage instances |
| **Model choice** | Pre-built foundation models only | Any model (custom, open-source, fine-tuned) |
| **Customization** | Prompt engineering, limited fine-tuning | Full control over architecture and training |
| **Latency** | Variable (shared infrastructure) | Predictable (dedicated instances) |
| **Cost model** | Per token (on-demand) | Per instance-hour (always running) |
| **Scale to zero** | Yes (pay only when invoked) | No (endpoint runs continuously) |
| **Data privacy** | Data processed on shared AWS infrastructure | Data stays on your dedicated instances |
| **Best for** | Text generation, summarization, Q&A via prompting | Custom models, specialized architectures, high-volume inference |

**Key decision points:**

| Scenario | Recommendation |
|----------|---------------|
| Prototyping a text generation feature | Bedrock (fast to start, no infrastructure) |
| High-volume production LLM inference (100K+ requests/day) | Cost comparison needed -- SageMaker or Bedrock provisioned |
| Custom-trained XGBoost fraud classifier | SageMaker (Bedrock does not host custom ML models) |
| Multi-modal application (text + image + code) | Bedrock (access to multi-modal foundation models) |
| Strict latency SLA (< 100ms p99) | SageMaker (dedicated instances) |

The two services are **complementary**: use Bedrock for text generation and summarization, SageMaker for custom classification and scoring.

> **Discussion:** For FraudShield's XGBoost fraud classifier scoring 500 TPS, would you use Bedrock or SageMaker? Why?

## Streaming Responses

When generating long responses, `invoke_model` waits for the entire output before returning. For interactive applications, this creates a poor user experience -- the user stares at a blank screen for several seconds.

**Streaming** sends tokens as they are generated, so the user sees output appearing incrementally.

| Aspect | `invoke_model` | `invoke_model_with_response_stream` |
|--------|---------------|-------------------------------------|
| Response delivery | All at once after generation completes | Token by token as generated |
| Latency to first token | High (waits for full generation) | Low (streams immediately) |
| Use case | Backend processing, batch jobs | Interactive applications, chat interfaces |
| Total generation time | Same | Same |

Streaming does not change the total generation time. It changes the **perceived** latency.

> **What is happening in the next cell:** We invoke Claude with `invoke_model_with_response_stream` and print each token as it arrives. Watch the output appear progressively rather than all at once.

In [ ]:
stream_prompt = f"""You are a fraud investigation analyst. Generate a brief (4-5 sentence) investigation summary for this flagged transaction:

- Amount: ${transaction['amount']:,.2f}
- Time: {transaction['hour']}:00
- Merchant: {transaction['merchant_name']}
- Merchant Risk Score: {transaction['merchant_risk_score']}
- International: {transaction['is_international']}
- Model Fraud Probability: {transaction['model_fraud_probability']}

Write the summary now."""

stream_request = json.dumps({
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 512,
    "temperature": 0.2,
    "messages": [
        {"role": "user", "content": stream_prompt}
    ]
})

print("Streaming response (tokens appear as generated):")
print("=" * 60)

start_time = time.time()
stream_response = bedrock_runtime.invoke_model_with_response_stream(
    modelId=MODEL_ID,
    contentType="application/json",
    accept="application/json",
    body=stream_request,
)

first_token_time = None
full_response = ""

for event in stream_response["body"]:
    chunk = json.loads(event["chunk"]["bytes"])
    if chunk["type"] == "content_block_delta":
        text = chunk["delta"].get("text", "")
        if text and first_token_time is None:
            first_token_time = time.time()
        print(text, end="", flush=True)
        full_response += text

total_time = time.time() - start_time
ttft = first_token_time - start_time if first_token_time else total_time

print(f"\n\n{'=' * 60}")
print(f"Time to first token: {ttft:.2f}s")
print(f"Total generation time: {total_time:.2f}s")
print(f"\nWith invoke_model, the user would wait {total_time:.2f}s before seeing anything.")
print(f"With streaming, the first token appeared after {ttft:.2f}s.")

## When to Use Bedrock vs SageMaker for LLMs

This is the decision framework for **LLM workloads specifically** (not custom ML models like XGBoost, which always belong on SageMaker).

| Scenario | Recommendation | Reasoning |
|----------|---------------|-----------|
| Prototyping a text generation feature | Bedrock | Fast to start, no infrastructure setup |
| High-volume production (100K+ req/day) | Evaluate both | Run a cost comparison at your volume |
| Need fine-tuned model on proprietary data | SageMaker or Bedrock custom model | SageMaker for full control, Bedrock for convenience |
| Multi-modal (text + image + code) | Bedrock | Access to multi-modal foundation models |
| Strict latency SLA (< 100ms p99) | SageMaker | Dedicated instances, predictable performance |
| Regulatory: data on dedicated infrastructure | SageMaker | Your instances, your VPC |

Many production architectures use **both**: Bedrock for text generation/summarization, SageMaker for custom classification and real-time scoring.

> **Discussion:** If FraudShield needs both fraud scoring (XGBoost, 500 TPS) and investigation summaries (LLM, 100/day), what architecture would you recommend?

---
# STAGE 3 -- Governance and Guardrails

You can now invoke foundation models and generate useful output. But in an enterprise, **access without governance is a liability**. Stage 3 covers the controls that make Bedrock production-ready: IAM policies, content filtering guardrails, PII redaction, cost management, monitoring, and throttling.

**What you will learn:**
- How to control Bedrock access with IAM policies
- How to create and test guardrails for content filtering and PII redaction
- How to manage costs with on-demand vs provisioned throughput
- How to monitor Bedrock usage with CloudWatch
- How to handle API throttling with exponential backoff

## IAM Policies for Bedrock

Bedrock access is controlled through standard IAM policies. The key permissions:

| Permission | Action | Use Case |
|------------|--------|----------|
| `bedrock:ListFoundationModels` | List available models | Discovery, model catalog |
| `bedrock:GetFoundationModel` | Get details about a specific model | Model selection |
| `bedrock:InvokeModel` | Send a prompt and get a response | Core inference |
| `bedrock:InvokeModelWithResponseStream` | Stream responses | Interactive applications |
| `bedrock:CreateGuardrail` | Create content filtering rules | Governance setup |
| `bedrock:UpdateGuardrail` | Modify existing guardrails | Policy changes |
| `bedrock:ApplyGuardrail` | Apply a guardrail to content | Content moderation |
| `bedrock:CreateProvisionedModelThroughput` | Reserve dedicated capacity | Production workloads |

**Principle of least privilege:**

| Role | Permissions | Rationale |
|------|-----------|----------|
| Fraud analyst | `InvokeModel` on Haiku only | Can generate summaries but not access expensive models |
| ML engineer | `InvokeModel`, `ListFoundationModels` | Can experiment with models |
| Governance admin | `CreateGuardrail`, `UpdateGuardrail` | Manages content policies but does not invoke models |
| Platform admin | `bedrock:*` | Full control |

**Resource-level control:** You can restrict access to specific models by ARN. This means you can allow Haiku but deny Opus -- controlling cost at the IAM level.

```json
{
  "Effect": "Allow",
  "Action": ["bedrock:InvokeModel"],
  "Resource": "arn:aws:bedrock:*::foundation-model/anthropic.claude-3-haiku*"
}
```

## Create a Guardrail

Guardrails are Bedrock's mechanism for **content filtering and PII protection**. A guardrail is a reusable policy that inspects both the input prompt and the output response.

**What a guardrail can do:**

| Component | What It Does | FraudShield Relevance |
|-----------|-------------|----------------------|
| **Content filter** | Block harmful content (hate, violence, sexual, insults, misconduct) | Prevent offensive investigation summaries |
| **Denied topics** | Block specific subjects you define | Prevent using the tool for non-fraud queries |
| **PII detection** | Identify and redact PII (names, SSNs, cards, emails, phones) | Redact customer data from summaries |
| **Word filters** | Block specific words or phrases | Block profanity or competitor names |

**How it works:** The guardrail wraps the model invocation with pre-processing (inspect the prompt) and post-processing (inspect the response). If a violation is detected, the guardrail either blocks the request entirely or redacts the sensitive content.

> **What is happening in the next cell:** We create a guardrail named `fraudshield-guardrail` with content filtering at MEDIUM strength, a denied topic (investment advice), and PII detection configured to anonymize names, email addresses, phone numbers, SSNs, and credit card numbers.

In [ ]:
GUARDRAIL_NAME = f"fraudshield-guardrail-{TIMESTAMP}"

try:
    guardrail_response = bedrock.create_guardrail(
        name=GUARDRAIL_NAME,
        description="FraudShield investigation summary guardrail: content filtering and PII redaction",
        topicPolicyConfig={
            "topicsConfig": [
                {
                    "name": "InvestmentAdvice",
                    "definition": "Providing specific investment recommendations, stock picks, or financial planning advice",
                    "examples": [
                        "What stocks should I buy?",
                        "Should I invest in cryptocurrency?",
                        "Give me financial planning advice",
                    ],
                    "type": "DENY",
                }
            ]
        },
        contentPolicyConfig={
            "filtersConfig": [
                {"type": "SEXUAL", "inputStrength": "MEDIUM", "outputStrength": "MEDIUM"},
                {"type": "VIOLENCE", "inputStrength": "MEDIUM", "outputStrength": "MEDIUM"},
                {"type": "HATE", "inputStrength": "MEDIUM", "outputStrength": "MEDIUM"},
                {"type": "INSULTS", "inputStrength": "MEDIUM", "outputStrength": "MEDIUM"},
                {"type": "MISCONDUCT", "inputStrength": "MEDIUM", "outputStrength": "MEDIUM"},
            ]
        },
        sensitiveInformationPolicyConfig={
            "piiEntitiesConfig": [
                {"type": "NAME", "action": "ANONYMIZE"},
                {"type": "EMAIL", "action": "ANONYMIZE"},
                {"type": "PHONE", "action": "ANONYMIZE"},
                {"type": "US_SOCIAL_SECURITY_NUMBER", "action": "ANONYMIZE"},
                {"type": "CREDIT_DEBIT_CARD_NUMBER", "action": "ANONYMIZE"},
            ]
        },
        blockedInputMessaging="This request has been blocked by the FraudShield guardrail. The input contains content that violates our usage policy.",
        blockedOutputsMessaging="This response has been blocked by the FraudShield guardrail. The output contains content that violates our usage policy.",
    )

    GUARDRAIL_ID = guardrail_response["guardrailId"]
    GUARDRAIL_VERSION = guardrail_response.get("version", "DRAFT")

    print(f"Guardrail created successfully")
    print(f"  Name:    {GUARDRAIL_NAME}")
    print(f"  ID:      {GUARDRAIL_ID}")
    print(f"  Version: {GUARDRAIL_VERSION}")
    print(f"\nConfigured protections:")
    print(f"  - Content filtering: MEDIUM strength (sexual, violence, hate, insults, misconduct)")
    print(f"  - Denied topics: Investment advice")
    print(f"  - PII redaction: Names, emails, phones, SSNs, credit card numbers")

except Exception as e:
    print(f"Guardrail creation failed: {e}")
    print(f"\nCommon causes:")
    print(f"  - IAM role lacks bedrock:CreateGuardrail permission")
    print(f"  - Guardrail name already exists (must be unique)")
    GUARDRAIL_ID = None
    GUARDRAIL_VERSION = None

## Test the Guardrail -- PII Redaction

We test the guardrail by sending a prompt that contains personally identifiable information. The guardrail should detect and redact the PII before it reaches the model and in the response.

> **What is happening in the next cell:** We invoke the model with a prompt containing a customer name, email address, and credit card number. The guardrail intercepts the PII and anonymizes it. Compare the original prompt (with PII) to the model's response (with PII redacted).

In [ ]:
if GUARDRAIL_ID:
    pii_prompt = """Generate a fraud investigation summary for the following case:

Customer John Smith (email: john.smith@example.com, phone: 555-123-4567) 
reported unauthorized charges on card number 4532-1234-5678-9012. 
Three transactions totaling $4,250 were made at overseas electronics retailers 
between 2:00 AM and 4:00 AM local time. The customer's SSN on file is 123-45-6789.

Provide a brief summary with risk assessment and recommended action."""

    pii_request = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 512,
        "temperature": 0.2,
        "messages": [
            {"role": "user", "content": pii_prompt}
        ]
    })

    try:
        pii_response = bedrock_runtime.invoke_model(
            modelId=MODEL_ID,
            contentType="application/json",
            accept="application/json",
            body=pii_request,
            guardrailIdentifier=GUARDRAIL_ID,
            guardrailVersion=GUARDRAIL_VERSION,
        )
        pii_response_body = json.loads(pii_response["body"].read())

        print("ORIGINAL PROMPT (contains PII):")
        print(f"  Name:  John Smith")
        print(f"  Email: john.smith@example.com")
        print(f"  Phone: 555-123-4567")
        print(f"  Card:  4532-1234-5678-9012")
        print(f"  SSN:   123-45-6789")
        print()

        if "content" in pii_response_body and pii_response_body["content"]:
            output_text = pii_response_body["content"][0]["text"]
            print("MODEL RESPONSE (guardrail applied):")
            print(output_text)
        else:
            print("GUARDRAIL BLOCKED THE REQUEST")
            print(json.dumps(pii_response_body, indent=2))

        guardrail_trace = pii_response_body.get("amazon-bedrock-guardrailAction", "none")
        print(f"\nGuardrail action: {guardrail_trace}")
        print("\nCheck the response above for redacted PII (replaced with placeholders).")
        print("This protects customer data in generated summaries -- critical for")
        print("GDPR, CCPA, PCI DSS, and other privacy regulations.")

    except Exception as e:
        print(f"Guardrail test failed: {e}")
        print("The guardrail may still be initializing. Wait 30 seconds and retry.")
else:
    print("Guardrail was not created. Skipping PII redaction test.")

## Test Content Filtering -- Denied Topic

We configured the guardrail to block investment advice. This tests the **denied topics** configuration.

> **What is happening in the next cell:** We send a prompt requesting investment advice. The guardrail should detect the denied topic and block the request entirely, returning the blocked input message instead of model output.

In [ ]:
if GUARDRAIL_ID:
    blocked_prompt = "Based on the fraud patterns you have analyzed, which stocks in the financial sector should I invest in to profit from increased fraud detection spending?"

    blocked_request = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 256,
        "temperature": 0.2,
        "messages": [
            {"role": "user", "content": blocked_prompt}
        ]
    })

    try:
        blocked_response = bedrock_runtime.invoke_model(
            modelId=MODEL_ID,
            contentType="application/json",
            accept="application/json",
            body=blocked_request,
            guardrailIdentifier=GUARDRAIL_ID,
            guardrailVersion=GUARDRAIL_VERSION,
        )
        blocked_response_body = json.loads(blocked_response["body"].read())

        print("PROMPT (denied topic -- investment advice):")
        print(f"  {blocked_prompt}")
        print()

        guardrail_action = blocked_response_body.get("amazon-bedrock-guardrailAction", "none")
        print(f"Guardrail action: {guardrail_action}")
        print()

        if "content" in blocked_response_body and blocked_response_body["content"]:
            response_text = blocked_response_body["content"][0]["text"]
            print(f"Response: {response_text}")
        else:
            print("Response blocked by guardrail.")
            print(json.dumps(blocked_response_body, indent=2))

        print("\nThe guardrail prevented the model from processing the off-topic request.")
        print("This is your first line of defense against prompt injection and misuse.")

    except Exception as e:
        print(f"Content filter test failed: {e}")
else:
    print("Guardrail was not created. Skipping content filtering test.")

## Cost Management: On-Demand vs Provisioned Throughput

Every Bedrock invocation has a cost. Understanding the pricing model is essential for budgeting.

### Pricing Models

| Pricing Model | How It Works | Best For |
|---------------|-------------|----------|
| **On-demand** | Pay per input/output token, no commitment | Variable workloads, prototyping, low-to-moderate volume |
| **Provisioned throughput** | Reserve dedicated capacity (model units), pay hourly | Predictable high-volume workloads, latency-sensitive applications |

### Illustrative On-Demand Pricing (check current AWS pricing for exact values)

| Model | Input (per 1K tokens) | Output (per 1K tokens) |
|-------|----------------------|------------------------|
| Claude 3 Haiku | ~$0.00025 | ~$0.00125 |
| Claude 3 Sonnet | ~$0.003 | ~$0.015 |
| Claude 3 Opus | ~$0.015 | ~$0.075 |
| Titan Text Express | ~$0.0002 | ~$0.0006 |

### Cost Example

FraudShield generates 10,000 investigation summaries per day. Each averages 500 input tokens and 300 output tokens.

| Model | Daily Input Cost | Daily Output Cost | Daily Total | Monthly Total |
|-------|-----------------|-------------------|-------------|---------------|
| Haiku | $1.25 | $3.75 | ~$5 | ~$150 |
| Sonnet | $15 | $45 | ~$60 | ~$1,800 |
| Opus | $75 | $225 | ~$300 | ~$9,000 |
| Titan | $1.00 | $1.80 | ~$3 | ~$90 |

**Model selection is a cost decision**, not just a quality decision. Start with the cheapest model that meets quality requirements.

> **Discussion:** When does provisioned throughput become cheaper than on-demand? (When sustained usage exceeds the break-even point. Also relevant when you need guaranteed latency -- on-demand has variable latency under load.)

## CloudWatch Monitoring for Bedrock

Bedrock publishes metrics to CloudWatch automatically. These metrics let you track usage, detect anomalies, and set up cost/performance alarms.

### Key Metrics

| Metric | What It Measures | Alert Threshold Example |
|--------|-----------------|------------------------|
| `Invocations` | Total number of model invocations | Spike above 2x baseline = unexpected usage |
| `InvocationLatency` | Time from request to complete response | p99 > 10s may indicate throttling |
| `InvocationClientErrors` | 4xx errors (bad requests, access denied) | Any sustained errors need investigation |
| `InvocationServerErrors` | 5xx errors (service issues) | Any errors warrant an AWS support case |
| `InputTokenCount` | Total input tokens consumed | Budget monitoring |
| `OutputTokenCount` | Total output tokens generated | Budget monitoring |

### Recommended Alarms

| Alarm | Purpose | Threshold |
|-------|---------|----------|
| **Daily token budget** | Prevent runaway costs | `InputTokenCount + OutputTokenCount > daily_limit` |
| **Error rate** | Detect configuration issues | `ClientErrors / Invocations > 5%` |
| **Latency spike** | Detect throttling or service issues | `p99 InvocationLatency > 15s` |

Set up a CloudWatch alarm on token counts with a daily threshold based on your budget. This prevents cost overruns from bugs, prompt injection attacks, or unexpected traffic spikes.

## API Quotas and Throttling

Bedrock has per-account, per-model, per-region rate limits. Exceeding them returns **HTTP 429 (ThrottlingException)**.

### Key Quotas

| Quota | Description | Typical Default |
|-------|-------------|----------------|
| **Requests per minute (RPM)** | API calls per minute | Varies by model (e.g., 100 RPM for some models) |
| **Tokens per minute (TPM)** | Total tokens processed per minute | Varies by model |
| **Max input tokens** | Maximum tokens in a single request | Model-dependent (e.g., 200K for Claude 3) |

### Retry Strategy: Exponential Backoff with Jitter

When throttled, do not retry immediately. Use exponential backoff with random jitter:

```python
import random

def invoke_with_retry(client, request_params, max_retries=5):
    base_delay = 1.0
    max_delay = 60.0
    
    for attempt in range(max_retries):
        try:
            return client.invoke_model(**request_params)
        except client.exceptions.ThrottlingException:
            if attempt == max_retries - 1:
                raise
            delay = min(base_delay * (2 ** attempt) + random.uniform(0, 1), max_delay)
            time.sleep(delay)
```

**Why jitter matters:** If 100 requests all get throttled and all retry after exactly 1 second, they all get throttled again (thundering herd). Random jitter spreads the retries over time.

**Quota increases:** You can request quota increases through AWS Service Quotas. For production workloads, plan ahead -- increases are not instant.

## Cleanup: Delete Guardrail

In production, guardrails persist as long-lived governance resources. For this training exercise, we clean up.

> **What is happening in the next cell:** We delete the guardrail we created. Bedrock does not have persistent endpoints or instances to delete -- only the guardrail resource needs cleanup.

In [ ]:
if GUARDRAIL_ID:
    try:
        bedrock.delete_guardrail(guardrailIdentifier=GUARDRAIL_ID)
        print(f"Deleted guardrail: {GUARDRAIL_NAME} (ID: {GUARDRAIL_ID})")
    except Exception as e:
        print(f"Guardrail cleanup: {e}")
else:
    print("No guardrail to delete.")

print("\nCleanup complete.")
print("\nBedrock does not have persistent endpoints or instances.")
print("On-demand invocations incur no ongoing cost when not in use.")
print("If you created provisioned throughput (we did not), you would delete that as well.")

---
## Course Wrap-up

This is the final teaching day of the AFD course. Here is what you accomplished across three weeks.

### Week 1: Foundations

| Topic | What You Learned |
|-------|----------------|
| ML Fundamentals | Supervised learning, evaluation metrics (precision, recall, F1, ROC-AUC), bias-variance trade-off |
| Neural Networks | Backpropagation, activation functions, loss functions, gradient descent |
| CNNs | Convolutional layers, pooling, feature extraction, transfer learning (MobileNet) |
| SageMaker Basics | Training jobs, model artifacts, S3 integration |

### Week 2: SageMaker + Advanced ML

| Topic | What You Learned |
|-------|----------------|
| SageMaker Pipelines & Registry | End-to-end MLOps: train, evaluate, register, deploy |
| Feature Store | Feature engineering, online/offline stores, point-in-time queries |
| Transformers | Attention from scratch, multi-head attention, positional encoding, BERT/GPT/T5 |
| HPO & Algorithms | Hyperparameter optimization, built-in algorithms, automatic model tuning |
| Inference Patterns | Real-time, serverless, async, batch transform, multi-model endpoints |
| Model Monitoring | Data capture, baselines, drift detection, EventBridge automation |

### Week 3 Tuesday (Today): Bedrock + Governance

| Topic | What You Learned |
|-------|----------------|
| Bedrock Foundations | Managed foundation model access, model selection, API invocation |
| Building with Bedrock | Structured prompts, model comparison, streaming responses |
| Governance | IAM policies, guardrails (content filtering, PII redaction), cost management, monitoring |

### The Full ML Spectrum

You now understand both sides of production ML:

| Approach | Service | When to Use |
|----------|---------|------------|
| **Build your own model** | SageMaker | Custom algorithms, domain-specific tasks, high-volume inference |
| **Use a foundation model** | Bedrock | Text generation, summarization, classification via prompting |
| **Both** | SageMaker + Bedrock | XGBoost for scoring + LLM for summaries (FraudShield pattern) |

### Review Preparation

The remaining days are open review and project time. Focus areas:

1. **ML Fundamentals:** Evaluation metrics, bias-variance, train/val/test splits
2. **Neural Networks and Deep Learning:** Backpropagation, CNNs, transformers, attention
3. **SageMaker Core:** Training jobs, model registry, endpoints, inference patterns
4. **SageMaker Advanced:** Feature Store, Experiments, HPO, monitoring, drift detection
5. **Bedrock:** Foundation models, model selection, guardrails, IAM, cost management
6. **MLOps:** The full lifecycle -- train, evaluate, register, deploy, monitor, retrain

Focus on understanding the **why** behind each service and technique, not memorizing API signatures. If you understand when to use Bedrock vs SageMaker, when to use serverless vs real-time inference, and how monitoring connects to retraining, you are well prepared.